In [1]:
import httpx
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
http_client = httpx.Client(verify=False)
client = Anthropic(http_client=http_client)

model = "claude-sonnet-4-6"

In [2]:
def classify_email(user_email):
    # 1. Define the System Prompt (Sets the persona and rules)
    system_prompt = "You are a customer support classifier. Your goal is to categorize emails into: BILLING, TECHNICAL, or GENERAL."

    # 2. Create the Few-Shot Prompt using XML tags
    # We provide 3 examples (shots) to show Claude the pattern.
    prompt = f"""
    Here are some examples of how to classify emails:

    <examples>
        <example>
            <input>I was charged twice for my subscription this month. Can you help?</input>
            <output>BILLING</output>
        </example>
        <example>
            <input>The app keeps crashing whenever I try to upload a photo.</input>
            <output>TECHNICAL</output>
        </example>
        <example>
            <input>What are your business hours on weekends?</input>
            <output>GENERAL</output>
        </example>
    </examples>

    Now, classify the following email:
    <email_to_classify>
    {user_email}
    </email_to_classify>

    Classification:"""

    # 3. Make the API call
    response = client.messages.create(
        model=model,
        max_tokens=1000,
        system=system_prompt,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.content[0].text

# Test the function
new_email = "My password reset link isn't working and I'm locked out."
result = classify_email(new_email)
print(f"Email: {new_email}")
print(f"Category: {result}")

Email: My password reset link isn't working and I'm locked out.
Category: **TECHNICAL**

This email describes a functional/access issue with the password reset system not working properly, which is a technical problem rather than a billing inquiry or general question.
